# Partial OT Active Regions Between Shapes

This notebook generates `fig:partial-ot-shape-active-mass`.

It solves the fixed-mass partial optimal transport problem
\[
\operatorname{POT}_m(a,b)=\min_{P\geq 0}\left\{\langle C,P\rangle:
P\mathbf 1\leq a,\;P^\top\mathbf 1\leq b,\;\mathbf 1^\top P\mathbf 1=m\right\}
\]
between two empirical indicator measures: a cat silhouette and an annulus. Both shapes are sampled by farthest-point sampling from their binary masks, then placed side by side with comparable visual scale. In each panel, pale points are available mass that is not transported by the optimal partial plan, while saturated points are the active source and target marginals selected by the plan.

In [1]:
from pathlib import Path
import os
import shutil
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

ROOT = Path.cwd()
if (ROOT / "notebooks-figures" / "figure_style.py").exists():
    FIGROOT = ROOT / "notebooks-figures"
elif (ROOT / "figure_style.py").exists():
    FIGROOT = ROOT
    ROOT = FIGROOT.parent
elif (ROOT.parent / "notebooks-figures" / "figure_style.py").exists():
    ROOT = ROOT.parent
    FIGROOT = ROOT / "notebooks-figures"
else:
    raise RuntimeError("Could not locate figure_style.py")

sys.path.insert(0, str(FIGROOT.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import ot
from PIL import Image

from figure_style import BLUE, RED, setup_matplotlib, figure_dir, interp_color, remove_axes, save_pdf

setup_matplotlib()
rng = np.random.default_rng(20260620)

NAME = "partial-ot-shape-active-mass"
OUT = figure_dir(NAME)
THUMB = FIGROOT / "thumbnails" / f"{NAME}.png"
THUMB.parent.mkdir(parents=True, exist_ok=True)


## Farthest-point samples of the binary masks

The masks are treated as indicator densities. Random candidates are drawn from each silhouette and then thinned by farthest-point sampling. This produces a dense but visually even empirical measure without imposing a square pixel grid on the display.

In [2]:
def load_mask_candidates(path, *, n_candidates=80000, seed=0):
    """Return random candidates inside a black-on-white binary silhouette."""
    local_rng = np.random.default_rng(seed)
    img = Image.open(path).convert("RGBA")
    arr = np.asarray(img, dtype=float) / 255.0
    luminance = arr[..., :3].mean(axis=2)
    mask = (arr[..., 3] > 0.25) & (luminance < 0.65)
    ys, xs = np.nonzero(mask)
    if len(xs) > n_candidates:
        idx = local_rng.choice(len(xs), n_candidates, replace=False)
        xs, ys = xs[idx], ys[idx]
    pts = np.column_stack([xs, -ys]).astype(float)
    pts += local_rng.uniform(-0.45, 0.45, size=pts.shape)
    pts -= pts.mean(axis=0, keepdims=True)
    pts /= np.max(np.linalg.norm(pts, axis=1))
    return pts


def farthest_point_sample(points, n):
    """Greedy farthest-point sampling from a candidate cloud."""
    start = int(np.argmin(np.sum(points**2, axis=1)))
    chosen = [start]
    dist2 = np.sum((points - points[start]) ** 2, axis=1)
    for _ in range(1, n):
        idx = int(np.argmax(dist2))
        chosen.append(idx)
        dist2 = np.minimum(dist2, np.sum((points - points[idx]) ** 2, axis=1))
    return points[np.asarray(chosen, dtype=int)]


n = 1200
cat_candidates = load_mask_candidates(FIGROOT / "assets" / "cat.png", seed=12)
annulus_candidates = load_mask_candidates(FIGROOT / "assets" / "annulus.png", seed=13)

x = farthest_point_sample(cat_candidates, n)
y = farthest_point_sample(annulus_candidates, n)

# Place the two shapes side by side. The raw annulus mask has a larger
# diameter than the cat after normalization, so it is rescaled to roughly
# match the cat height before shifting it to the right.
x = 1.00 * x + np.array([-0.78, 0.0])
y = 0.72 * y + np.array([0.82, -0.005])

all_points = np.vstack([x, y])
lo = all_points.min(axis=0)
hi = all_points.max(axis=0)
pad = np.array([0.14, 0.12])
limits = ((lo[0] - pad[0], hi[0] + pad[0]), (lo[1] - pad[1], hi[1] + pad[1]))


## Exact fixed-mass partial OT plans

The cost is the normalized squared Euclidean distance between the sampled points. The POT exact partial solver is used with two dummy atoms; this is still a partial transport plan, not an entropic approximation. The active source and target marginals are the row and column sums of the optimal subcoupling.

In [3]:
a = np.ones(n) / n
b = np.ones(n) / n
C = np.sum((x[:, None, :] - y[None, :, :]) ** 2, axis=2)
C = C / C.max()

masses = [0.82, 0.58, 0.36, 0.18]
plans = []
for mass in masses:
    P = ot.partial.partial_wasserstein(a, b, C, m=mass, nb_dummies=2, numItermax=2_000_000)
    if abs(P.sum() - mass) > 5e-9:
        raise RuntimeError(f"Partial plan mass {P.sum()} differs from requested mass {mass}.")
    plans.append(P)


## Geometry panels

Each panel displays the same two point clouds. Points not selected by the active marginals are kept with alpha `0.1`; the saturated red and blue points are those carrying transported mass. A small set of thin black segments is overlaid by first selecting farthest-spaced active source locations and then drawing the strongest outgoing transport edge from each selected source. This reveals the geometry of the partial plan without cluttering the active-region display.


In [4]:
def farthest_indices(points, k):
    """Return indices of a farthest-point subsample within `points`."""
    if len(points) == 0 or k <= 0:
        return np.array([], dtype=int)
    k = min(k, len(points))
    start = int(np.argmin(np.sum((points - points.mean(axis=0, keepdims=True)) ** 2, axis=1)))
    chosen = [start]
    dist2 = np.sum((points - points[start]) ** 2, axis=1)
    for _ in range(1, k):
        idx = int(np.argmax(dist2))
        chosen.append(idx)
        dist2 = np.minimum(dist2, np.sum((points - points[idx]) ** 2, axis=1))
    return np.asarray(chosen, dtype=int)


def representative_plan_edges(P, max_edges=16):
    """Subsample transported source locations and keep their strongest outgoing edge."""
    row = P.sum(axis=1)
    active_sources = np.flatnonzero(row > 1e-12)
    if len(active_sources) == 0:
        return []
    local = farthest_indices(x[active_sources], max_edges)
    edges = []
    for i in active_sources[local]:
        js = np.flatnonzero(P[i] > 1e-13)
        if len(js) == 0:
            continue
        j = int(js[np.argmax(P[i, js])])
        edges.append((int(i), j))
    return edges


def draw_plan_segments(ax, P):
    for i, j in representative_plan_edges(P):
        ax.plot(
            [x[i, 0], y[j, 0]],
            [x[i, 1], y[j, 1]],
            color="black",
            alpha=0.44,
            lw=0.28,
            solid_capstyle="round",
            zorder=2,
        )


def draw_active_panel(ax, P):
    row = P.sum(axis=1)
    col = P.sum(axis=0)
    source_active = row > 1e-12
    target_active = col > 1e-12

    # Inactive mass first: it shows the original support without competing with the active marginals.
    ax.scatter(x[~source_active, 0], x[~source_active, 1], s=3.0, color=RED, alpha=0.10, edgecolor="none", linewidth=0, zorder=1)
    ax.scatter(y[~target_active, 0], y[~target_active, 1], s=3.0, color=BLUE, alpha=0.10, edgecolor="none", linewidth=0, zorder=1)

    # Sparse black edges make the partial plan readable without covering the active supports.
    draw_plan_segments(ax, P)

    # Active submarginals use uniform small dots: the figure should show selected
    # support regions, not encode a second mass scale through marker area.
    ax.scatter(x[source_active, 0], x[source_active, 1], s=4.8, color=RED, alpha=0.92, edgecolor="none", linewidth=0, zorder=3)
    ax.scatter(y[target_active, 0], y[target_active, 1], s=4.8, color=BLUE, alpha=0.92, edgecolor="none", linewidth=0, zorder=4)

    ax.set_aspect("equal")
    ax.set_xlim(*limits[0])
    ax.set_ylim(*limits[1])
    remove_axes(ax)


def export_panel(P, mass):
    fig, ax = plt.subplots(figsize=(2.55, 1.42))
    draw_active_panel(ax, P)
    save_pdf(fig, OUT / f"mass-{int(round(100 * mass)):02d}.pdf", pad_inches=0.018)
    plt.close(fig)

for mass, P in zip(masses, plans):
    export_panel(P, mass)

# Flattened copies used by the arXiv export.
arxiv_figures = ROOT / "arxiv" / "figures"
arxiv_figures.mkdir(parents=True, exist_ok=True)
for mass in masses:
    src = OUT / f"mass-{int(round(100 * mass)):02d}.pdf"
    dst = arxiv_figures / f"{NAME}--mass-{int(round(100 * mass)):02d}.pdf"
    shutil.copyfile(src, dst)


## Thumbnail

The manuscript arranges the exported PDF panels directly and adds the transported-mass labels in LaTeX. The thumbnail below previews the same four-panel composition for the figure notebook gallery.

In [5]:
def make_thumbnail(path):
    fig, axes = plt.subplots(1, 4, figsize=(10.2, 2.0), constrained_layout=True)
    for ax, mass, P in zip(axes, masses, plans):
        draw_active_panel(ax, P)
        ax.text(0.5, 1.02, rf"$m={mass:.2f}$", transform=ax.transAxes, ha="center", va="bottom", fontsize=9)
    fig.savefig(path, dpi=180, bbox_inches="tight", pad_inches=0.06)
    plt.close(fig)

make_thumbnail(THUMB)


![Partial OT active regions between shapes](thumbnails/partial-ot-shape-active-mass.png)